In [ ]:
# =============================================================================
# 1. Importación de Librerías & Cliente
# =============================================================================
import pandas as pd
from datetime import datetime, timedelta
import pytz
from google.cloud import bigquery
from google.cloud import storage
clientBQ = bigquery.Client()
storage_client = storage.Client()

In [ ]:
# =============================================================================
# 2. Configuración de Fechas D-1 & Rutas
# =============================================================================

Zona = pytz.timezone('America/Lima')
peru_time = datetime.now(Zona)
peru_time_ayer = peru_time - timedelta(days=1)
var_fecha_ini = peru_time_ayer.strftime('%Y-%m-%d') # Para el WHERE del SQL
var_fecha_fin = peru_time.strftime('%Y-%m-%d')
fecha_fin_dt = datetime.strptime(var_fecha_fin, '%Y-%m-%d')

print(f"--- Fecha de inicio: {var_fecha_ini} ---")
print(f"--- Fecha de Fin: {var_fecha_fin} ---")
print(f"--- Fecha del proceso: {fecha_fin_dt} ---")

--- Fecha de inicio: 2026-02-16 ---
--- Fecha de Fin: 2026-02-17 ---
--- Fecha del proceso: 2026-02-17 00:00:00 ---


In [ ]:
## Variables de fecha como DataEntry
#var_fecha_ini = '2026-02-09'
#var_fecha_fin = '2026-02-10'
#fecha_fin_dt = datetime.strptime(var_fecha_fin, '%Y-%m-%d')

##print(f"--- Fecha de inicio: {var_fecha_ini} ---")
##print(f"--- Fecha de Fin: {var_fecha_fin} ---")
##print(f"--- Fecha del proceso: {fecha_fin_dt} ---")

In [ ]:
# =============================================================================
# 2. Configuración de Variables de entorno
# =============================================================================
var_anho = fecha_fin_dt.strftime('%Y')               # Para la carpeta
var_mes = fecha_fin_dt.strftime('%m')                # Para la carpeta
var_fecha_file = fecha_fin_dt.strftime('%Y%m%d')     # Para el nombre del archivo

print(f"--- Año del proceso {var_anho} ---")
print(f"--- Mes del proceso: {var_mes} ---")
print(f"--- Fecha del archivo: {var_fecha_file} ---")


--- Año del proceso 2026 ---
--- Mes del proceso: 02 ---
--- Fecha del archivo: 20260217 ---


In [ ]:
# =============================================================================
# 3. Configuración de Rutas y Parámetros
# =============================================================================
bucket_name = "adls-reportes"
ruta_base = f"Data/APA/abono_finanzas/{var_anho}/{var_mes}/"
nombre_final = f"v_abono_finanzas_{var_fecha_file}.parquet"


In [ ]:
# =============================================================================
# 4. Ejecución de Exportación
# =============================================================================

# Nombre temporal con comodín (obligatorio para BigQuery)
uri_temporal = f"gs://{bucket_name}/{ruta_base}temp_{var_fecha_file}_*.parquet"
print(f"--- Iniciando proceso para la fecha: {var_fecha_ini} ---")

query_export = f"""
EXPORT DATA OPTIONS (
  uri = '{uri_temporal}',
  format = 'PARQUET',
  overwrite = true
) AS
SELECT
  flujo,
  producto,
  cod_comercio,
  cod_trx,
  des_trx,
  fecha_proceso,
  cod_banco,
  tipo_cuenta,
  num_cuenta,
  cod_moneda,
  abono_neto,
  abono_neto_dolar,
  tipo_cambio,
  abono_neto_solarizado,
  importe,
  comision_abono,
  comision_procesamiento,
  cobro_devolucion,
  importe_retenido,
  fecha_abono_prog,
  tipo_doc,
  document_number,
  situacion,
  autorizacion_trx,
  cod_moneda_trx,
  cod_banco_trx,
  imp_txn,
  imp_impu_txn,
  importe_abonar_trx,
  acumulado_neto_trx,
  saldo_restante_trx,
  num_txn,
  num_txn_restante,
  banco_rechazo,
  estado_rechazo,
  motivo_rechazo,
  fecha_rechazo,
  nom_comercio
FROM `prd-izipay-data-storage-pv.bi_apa.v_abono_finanzas`
WHERE fecha_proceso >= DATE '{var_fecha_ini}'
and fecha_proceso < DATE '{var_fecha_fin}';
"""

print("Paso 1: Exportando partes desde BigQuery...")
clientBQ.query(query_export).result()

--- Iniciando proceso para la fecha: 2026-02-16 ---
Paso 1: Exportando partes desde BigQuery...


In [ ]:
# =============================================================================
# 5. Consolidación de partes en un solo archivo con dataframe
# =============================================================================

# Consolidando en un solo archivo

print(" Consolidando archivos en uno solo...")
bucket = storage_client.bucket(bucket_name)
prefix_temp = f"{ruta_base}temp_{var_fecha_file}_"

# Listamos todas las partes generadas
blobs = list(bucket.list_blobs(prefix=prefix_temp))

if not blobs:
    print("⚠️ AVISO: No se encontraron archivos temporales para consolidar. Proceso continúa sin limpieza.")
else:
    # Leemos cada parte y las unimos en memoria con Pandas
    lista_df = []
    for blob in blobs:
        uri_parte = f"gs://{bucket_name}/{blob.name}"
        lista_df.append(pd.read_parquet(uri_parte))

    df_final = pd.concat(lista_df, ignore_index=True)

    # Guardamos el DataFrame consolidado como el archivo final solicitado
    ruta_final_full = f"gs://{bucket_name}/{ruta_base}{nombre_final}"
    df_final.to_parquet(ruta_final_full, index=False)

    # 6. Limpieza: Borrar los archivos temporales
    for blob in blobs:
        try:
            bucket.blob(blob.name).delete()
        except Exception as e:
            print(f"⚠️ AVISO: No se pudo borrar {blob.name}: {e}")

    print(f"✅ ÉXITO: Archivo único creado en: {ruta_final_full}")

# Limpieza final: eliminar temporales que empiecen con "temp"
prefix_temp_all = f"{ruta_base}temp"
blobs_temp_all = list(bucket.list_blobs(prefix=prefix_temp_all))

if blobs_temp_all:
    bucket.delete_blobs(blobs_temp_all)
    print(f"🧹 Eliminados {len(blobs_temp_all)} archivos temporales finales")
else:
    print("No se encontraron temporales finales")


 Consolidando archivos en uno solo...
✅ ÉXITO: Archivo único creado en: gs://adls-reportes/Data/APA/abono_finanzas/2026/02/v_abono_finanzas_20260217.parquet
No se encontraron temporales previos
